# Visualization — Interactive Reports & Charts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/05_visualization.ipynb)

**What this does:** Creates publication-quality interactive Plotly visualizations and static matplotlib charts for pathway subtyping results. Includes dimensionality reduction (UMAP, t-SNE, PCA), heatmaps, radar charts, and self-contained HTML reports.

**Two modes:**
- **Interactive** (`pip install pathway-subtyping[viz]`) — Plotly charts with hover, zoom, export
- **Static** (base install) — matplotlib charts, no extra dependencies

**Prerequisites:** [00_quick_demo.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/00_quick_demo.ipynb)

In [ ]:
# Install with visualization extras (Plotly + kaleido for export)
!pip install -q "pathway-subtyping[viz]==0.3.0"

import pathway_subtyping
print(f"pathway-subtyping v{pathway_subtyping.__version__}")

# Check Plotly availability
try:
    import plotly
    print(f"Plotly v{plotly.__version__} — interactive mode available")
except ImportError:
    print("Plotly not installed — static mode only")

## 1. Generate Example Data

We'll create a synthetic cohort and cluster it to have data for visualization.

In [ ]:
from pathway_subtyping import (
    SimulationConfig,
    generate_synthetic_data,
    run_clustering,
    ClusteringAlgorithm,
)
import numpy as np

sim = generate_synthetic_data(SimulationConfig(
    n_samples=200,
    n_pathways=12,
    n_genes_per_pathway=20,
    n_subtypes=3,
    effect_size=1.2,
    noise_level=1.0,
    seed=42,
))

clustering = run_clustering(
    sim.pathway_scores.values,
    n_clusters=3,
    algorithm=ClusteringAlgorithm.GMM,
    seed=42,
)

print(f"Data: {sim.pathway_scores.shape[0]} samples × {sim.pathway_scores.shape[1]} pathways")
print(f"Clusters: {len(set(clustering.labels))} subtypes")
print(f"Silhouette: {clustering.silhouette:.3f}")

## 2. Interactive Scatter Plots (UMAP, t-SNE, PCA)

Dimensionality reduction projects high-dimensional pathway scores into 2D for visual inspection. Each point is a sample, colored by cluster.

In [ ]:
from pathway_subtyping import plot_interactive_scatter, DimReductionMethod

# UMAP (best for preserving local structure)
fig_umap = plot_interactive_scatter(
    sim.pathway_scores,
    clustering.labels,
    method=DimReductionMethod.UMAP,
    title="UMAP — Pathway Subtypes",
    seed=42,
)
fig_umap.show()

In [ ]:
# t-SNE (good for cluster separation)
fig_tsne = plot_interactive_scatter(
    sim.pathway_scores,
    clustering.labels,
    method=DimReductionMethod.TSNE,
    title="t-SNE — Pathway Subtypes",
    seed=42,
)
fig_tsne.show()

In [ ]:
# PCA (linear, fastest, shows variance explained)
fig_pca = plot_interactive_scatter(
    sim.pathway_scores,
    clustering.labels,
    method=DimReductionMethod.PCA,
    title="PCA — Pathway Subtypes",
    seed=42,
)
fig_pca.show()

## 3. Interactive Heatmap

Shows mean pathway scores per cluster. Reveals which pathways distinguish each subtype.

In [ ]:
from pathway_subtyping import plot_interactive_heatmap

fig_heatmap = plot_interactive_heatmap(
    sim.pathway_scores,
    clustering.labels,
    title="Pathway Profiles by Subtype",
)
fig_heatmap.show()

## 4. Radar Chart — Subtype Pathway Profiles

Radar (spider) charts show the top pathways for each subtype overlaid, making it easy to see distinctive profiles.

In [ ]:
from pathway_subtyping import plot_subtype_trajectories

fig_radar = plot_subtype_trajectories(
    sim.pathway_scores,
    clustering.labels,
    top_n=8,
    title="Subtype Pathway Profiles",
)
fig_radar.show()

## 5. Cluster Distribution

Bar chart showing the number of samples in each cluster.

In [ ]:
from pathway_subtyping import plot_cluster_distribution

fig_dist = plot_cluster_distribution(
    clustering.labels,
    title="Cluster Size Distribution",
)
fig_dist.show()

## 6. Full Interactive HTML Report

Generate a self-contained HTML file with all charts combined. Open it in any browser — no Python needed to view.

In [ ]:
from pathway_subtyping import create_interactive_report, ReportConfig, DimReductionMethod

config = ReportConfig(
    title="Synthetic Cohort Analysis",
    dim_reduction=DimReductionMethod.UMAP,
    disclaimer="Demonstration with synthetic data. Not for clinical use.",
)

report = create_interactive_report(
    pathway_scores=sim.pathway_scores,
    labels=clustering.labels,
    output_path="report.html",
    config=config,
    seed=42,
)

print(f"Report saved to: report.html")
print(f"Sections: {report.n_sections}")

# In Colab, display download link
try:
    from google.colab import files
    files.download("report.html")
except ImportError:
    print("Open report.html in your browser to view the interactive report.")

## 7. Static Fallback (No Plotly Required)

These charts work with just matplotlib — included in the base install.

In [ ]:
from pathway_subtyping import plot_static_scatter, DimReductionMethod

fig = plot_static_scatter(
    sim.pathway_scores,
    clustering.labels,
    method=DimReductionMethod.PCA,
    title="PCA — Static Plot",
    seed=42,
)

## 8. Exporting Figures

Export any Plotly figure to PNG, SVG, PDF, or HTML.

In [ ]:
from pathway_subtyping import export_figure, FigureFormat

# Export the UMAP scatter in multiple formats
export_figure(fig_umap, "umap_scatter", [FigureFormat.HTML, FigureFormat.PNG, FigureFormat.SVG])
print("Exported: umap_scatter.html, umap_scatter.png, umap_scatter.svg")

# Export heatmap as HTML only
export_figure(fig_heatmap, "heatmap", [FigureFormat.HTML])
print("Exported: heatmap.html")

## Summary

| Chart | Function | Best For |
|-------|----------|----------|
| UMAP/t-SNE/PCA scatter | `plot_interactive_scatter()` | Cluster separation overview |
| Heatmap | `plot_interactive_heatmap()` | Pathway profiles per subtype |
| Radar chart | `plot_subtype_trajectories()` | Comparing subtype signatures |
| Distribution | `plot_cluster_distribution()` | Cluster balance check |
| Full report | `create_interactive_report()` | Sharing with collaborators |
| Static scatter | `plot_static_scatter()` | Environments without Plotly |

## Next Steps

- **Expression scoring:** [02_expression_scoring.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/02_expression_scoring.ipynb)
- **Multi-omic fusion:** [03_multi_omic_fusion.ipynb](https://colab.research.google.com/github/topmist-admin/pathway-subtyping-framework/blob/main/examples/notebooks/03_multi_omic_fusion.ipynb)
- **API reference:** [Visualization API](https://github.com/topmist-admin/pathway-subtyping-framework/blob/main/docs/api/visualization.md)

---
*Built with [pathway-subtyping](https://pypi.org/project/pathway-subtyping/). Disease-agnostic. Open source.*